In [1]:
import pickle
import warnings
import json
warnings.filterwarnings("ignore")

import pandas as pd
import numpy  as np

from sklearn.model_selection    import train_test_split
from pathlib                    import Path
from IPython.display            import display, HTML
from methods.FeatureSelection import (
                                        remove_low_variance_features, 
                                        nan_values, 
                                        const_feature,
                                        fill_missing_values,
                                        get_feature_intervals,
                                        compare_samples
                                    )

In [2]:
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
display(HTML("<style>.container { width:99% !important; }</style>"))
pd.options.display.float_format = '{:,.3f}'.format
pd.set_option('display.max_columns', None)
# ширина каждого столбца (в символах)
pd.set_option('display.max_colwidth', None)

In [3]:
path = f'prepared_data/dataset_for_feature_selection.pickle'

# загрузка данных
with open(path, 'rb') as f:
    samples = pickle.load(f)

# проверим, какие ключи есть в загруженном словаре
print("Ключи в samples:", samples.keys())

Ключи в samples: dict_keys(['df', 'info_fields', 'features', 'cat_features', 'num_features', 'target'])


In [4]:
df           = samples['df']
info         = samples['info_fields']
cat_features = samples['cat_features']
num_features = samples['num_features']
target       = samples['target']

## Анализ вещественных фичей

In [5]:
feature_reasons = {}
low_var_features                         = remove_low_variance_features(df, num_features, threshold=0.001)
feature_reasons['low_variance_features'] = low_var_features
num_features                             = list(set(num_features) - set(low_var_features))

open                            153,796,939.973
high                            159,071,506.652
low                             149,197,597.594
close                           151,721,153.008
volume           15,947,415,700,913,983,488.000
ret_1                                     0.000
ret_3                                     0.000
ret_5                                     0.002
ret_20                                    0.012
vol3                                      0.000
vol_5                                     0.000
vol_15                                    0.000
vol_30                                    0.000
vol_45                                    0.000
vol_90                                    0.000
mom_3                                     0.001
mom_5                                     0.002
mom_15                                    0.008
mom_20                                    0.012
sma_3                           152,732,585.798
sma_5                           152,823,

In [6]:
# удаление константных фичей
const_features                    = const_feature(df, num_features,  0.9)
num_features                      = list(set(num_features) - set(const_features))
feature_reasons['const_features'] = const_features

In [7]:
# удаление фичей с пропусками
nan_features                    = nan_values(df, num_features, 0.2)
num_features                    = list(set(num_features) - set(nan_features))
feature_reasons['nan_features'] = nan_features

# кол-во пропусков по данным фичам
if nan_features:
    (df[nan_features].isna().sum() / df.shape[0]).sort_values(ascending=False)

In [8]:
reasons = []
for reason, features in feature_reasons.items():
    for feature, explanation in features.items():
        reasons.append({
            "reason"     : reason,
            "feature"    : feature,
            "explanation": explanation
        })

reasons_df = pd.DataFrame(reasons)

In [9]:
reasons_df

,reason,feature,explanation
0,low_variance_features,ret_1,0.000
1,low_variance_features,ret_3,0.000
2,low_variance_features,vol3,0.000
3,low_variance_features,vol_5,0.000
4,low_variance_features,vol_15,0.000
5,low_variance_features,vol_30,0.000
6,low_variance_features,vol_45,0.000
7,low_variance_features,vol_90,0.000
8,low_variance_features,hl_range,0.001
9,const_features,is_month_start,"Мажоритарное значение 0, доля значения - 0.971"


In [10]:
# очищаем лишние поля
features = num_features + cat_features
# df = df[info + features + [target]]

# Удаляем записи, в которых остались пропуски

In [11]:
df.isna().sum()

date                 0
open                 0
high                 0
low                  0
close                0
volume               0
ticker               0
ret_1               69
ret_3               69
ret_5              345
ret_20            1380
vol3               207
vol_5              345
vol_15            1035
vol_30            2070
vol_45            3090
vol_90            6150
mom_3              207
mom_5              345
mom_15            1035
mom_20            1380
sma_3              138
sma_5              276
sma_10             621
sma_20            1311
sma_ratio         1311
vol_mean_5         276
vol_mean_15        966
vol_mean_45       3022
vol_ratio          276
hl_range             0
dow                  0
month                0
week                 0
is_month_end         0
is_month_start       0
target               0
dtype: int64

In [12]:
df = df.dropna()

In [13]:
df.isna().sum()

date              0
open              0
high              0
low               0
close             0
volume            0
ticker            0
ret_1             0
ret_3             0
ret_5             0
ret_20            0
vol3              0
vol_5             0
vol_15            0
vol_30            0
vol_45            0
vol_90            0
mom_3             0
mom_5             0
mom_15            0
mom_20            0
sma_3             0
sma_5             0
sma_10            0
sma_20            0
sma_ratio         0
vol_mean_5        0
vol_mean_15       0
vol_mean_45       0
vol_ratio         0
hl_range          0
dow               0
month             0
week              0
is_month_end      0
is_month_start    0
target            0
dtype: int64

In [14]:
# приводим все категориальные фичи к строковому типу
df[cat_features] = df[cat_features].astype('str')

## Разбиваем на подвыборки

In [15]:
# очищаем лишние поля которые могли оставться в датафрейме послое фичи селекшена
merged_list = info + num_features + cat_features + [target]
result_df   = df[merged_list].copy()
print('Размер: ', result_df.shape)
result_df.head(2)

Размер:  (75535, 26)


,date,ticker,sma_3,close,sma_5,vol_mean_5,mom_3,month,week,mom_15,ret_20,low,ret_5,sma_10,vol_mean_45,volume,sma_ratio,mom_5,dow,high,vol_mean_15,open,mom_20,sma_20,vol_ratio,target
90,2020-05-15,AFKS,13.987,14.000,14.170,"20,539,620.000",-0.028,5,20.000,0.025,0.063,13.904,-0.027,14.290,"52,173,768.889","14,333,100.000",0.014,-0.027,4.000,14.070,"25,249,506.667",13.980,0.063,13.978,0.698,1
91,2020-05-18,AFKS,14.073,14.420,14.157,"25,255,360.000",0.018,5,21.000,0.043,0.109,14.001,-0.004,14.287,"50,913,982.222","38,631,800.000",0.008,-0.004,0.000,14.449,"26,083,960.000",14.036,0.109,14.049,1.530,1


In [16]:
build, test  = train_test_split(result_df, stratify=result_df[target], test_size=0.2, random_state=42)
train, valid = train_test_split(build, stratify=build[target], test_size=0.2, random_state=42)

## Сравнение выборок на схожесть фичей

In [17]:
# колонки для сравнения compare
features_to_compare = num_features + cat_features

In [18]:
train_valid_comparison_df = compare_samples(train, valid, features_to_compare).reset_index().rename({'index': 'feature'}, axis = 1)[['feature', 'euclidean']].sort_values(by = 'euclidean', ascending = False)
build_test_comparison_df  = compare_samples(build, test, features_to_compare).reset_index().rename({'index': 'feature'}, axis = 1)[['feature', 'euclidean']].sort_values(by = 'euclidean', ascending = False)

In [19]:
train_valid_comparison_df

,feature,euclidean
5,month,100.000
3,vol_mean_5,99.616
22,vol_ratio,99.082
6,week,98.975
4,mom_3,98.752
10,ret_5,98.532
15,mom_5,98.532
17,high,98.274
11,sma_10,98.261
21,sma_20,98.227


In [20]:
build_test_comparison_df

,feature,euclidean
5,month,100.000
22,vol_ratio,99.521
10,ret_5,99.267
15,mom_5,99.267
14,sma_ratio,99.250
7,mom_15,99.067
4,mom_3,99.059
20,mom_20,98.982
8,ret_20,98.982
6,week,98.975


In [21]:
drop_features = []

train = train.drop(drop_features, axis = 1, errors = 'ignore')
valid = valid.drop(drop_features, axis = 1, errors = 'ignore')
test  = test.drop(drop_features, axis = 1, errors = 'ignore')

cat_features = [
    col for col in train.drop(info + [target], axis = 1).columns.to_list()
    if col in train.columns and train[col].dtype == "object"
]

In [22]:
train.shape,valid.shape, test.shape, build.shape

((48342, 26), (12086, 26), (15107, 26), (60428, 26))

In [23]:
#словарь с ключами:
#      cat_features
#  - unique_list уникальные значение на build выборки
#  - freq_value частовстречаемое значение на build выборки
#  - default_value значение, на которое нужно заменить категорию, которая не вошла в unique_list
# 
#      num_features
#  - min
#  - max
#  - mean



features_intervals = get_feature_intervals(build[num_features + cat_features])
features_intervals

{'sma_3': {'min': 0.01, 'max': 112218.0, 'mean': 2478.488},
 'close': {'min': 0.01, 'max': 111838.0, 'mean': 2473.047},
 'sma_5': {'min': 0.01, 'max': 112202.0, 'mean': 2481.21},
 'vol_mean_5': {'min': 2021.28, 'max': 39736570800.0, 'mean': 705209890.142},
 'mom_3': {'min': -0.119, 'max': 0.122, 'mean': 0.001},
 'month': {'min': 1.0, 'max': 12.0, 'mean': 6.906},
 'week': {'min': 1.0, 'max': 52.0, 'mean': 28.246},
 'mom_15': {'min': -0.281, 'max': 0.285, 'mean': 0.004},
 'ret_20': {'min': -0.312, 'max': 0.33, 'mean': 0.005},
 'low': {'min': 0.009, 'max': 111050.0, 'mean': 2444.824},
 'ret_5': {'min': -0.157, 'max': 0.161, 'mean': 0.001},
 'sma_10': {'min': 0.01, 'max': 114062.35, 'mean': 2506.698},
 'vol_mean_45': {'min': 2372.014,
  'max': 44334084142.222,
  'mean': 812607781.705},
 'volume': {'min': 1799.68, 'max': 34841767600.0, 'mean': 632140655.959},
 'sma_ratio': {'min': -0.16, 'max': 0.126, 'mean': -0.0},
 'mom_5': {'min': -0.157, 'max': 0.161, 'mean': 0.001},
 'dow': {'min': 0.0

In [24]:
# пришла пустота на инференс ты меняешь на наиболее вероятное/популярное (для категориальной)  для числовой на среднее
# если числовая пришла выше границы, то замечанием на верзнюю границу если нижнняя на нижнюю

# если пришло значение но его нет в словаре заменяем на other
# как быть если не обучалась на other но пришло на инфуренс категориальное значение которго не было на обучении
# заменять на наиболее вероятное

#  замена новых значений в test 
for feature in features_intervals.keys():
    if feature in cat_features:
        test.loc[test[feature].isna(), feature] = features_intervals[feature]['freq_value']
        test.loc[~ test[feature].isin(features_intervals[feature]['unique_list']), feature] = features_intervals[feature]['default_value']
    else:
        test.loc[test[feature].isna(), feature] = features_intervals[feature]['mean']
        test.loc[test[feature]>features_intervals[feature]['max'], feature] = features_intervals[feature]['max']
        test.loc[test[feature]<features_intervals[feature]['min'], feature] = features_intervals[feature]['min']

In [25]:
path = f'prepared_data/data_for_modeling.pickle'

samples = {
    'train'       : train,
    'valid'       : valid,
    'test'        : test,
    'target'      : target,
    'reasons_df'  : reasons_df,
    'info_fields' : info,
    'features'    : train.drop(info + [target], axis = 1).columns.to_list(),
    'cat_features': cat_features,
    'features_intervals' : features_intervals
}
with open(path, 'wb') as f:
    pickle.dump(samples, f)